# Medical MLP Classification

This notebook summarizes the Pima diabetes MLP experiment. The reusable experiment code lives in `src/medical_mlp_classification/experiment.py`; the command-line runner is `scripts/run_experiment.py`.

## Experiment Setup

The dataset is loaded from LIBSVM sparse format with feature indices preserved. Raw label `-1` is mapped to `diabetes_positive=1`, matching the diabetes-positive minority class in this dataset. The downloaded data file is checked against a pinned SHA-256 digest.

Models are ranked by validation AUC-ROC. Each model's decision threshold is selected on validation F1, then applied once to the held-out test split.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RESULTS = ROOT / "results"

summary_path = RESULTS / "summary.json"
metrics_path = RESULTS / "model_comparison.csv"
summary_path, metrics_path

(WindowsPath('C:/Users/su109/AppData/Local/Temp/codex-repo-reviews/mlp-medical-classification-20260617221340/results/summary.json'),
 WindowsPath('C:/Users/su109/AppData/Local/Temp/codex-repo-reviews/mlp-medical-classification-20260617221340/results/model_comparison.csv'))

## Dataset Metadata

The committed `summary.json` records the dataset size, label mapping, source URL, and data checksum used by the experiment.

In [2]:
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary["dataset"]

{'rows': 768,
 'features': 8,
 'raw_label_counts': {'-1': 268, '1': 500},
 'target_counts': {'0': 500, '1': 268},
 'positive_label_mapping': 'raw LIBSVM label -1 -> diabetes_positive=1',
 'data_url': 'https://www.csie.ntu.edu.tw/~cjlin/libsvmtools/datasets/binary/diabetes_scale',
 'data_sha256': '0c07eb4c49e7a8ffb9c9f25095ac3022df2ca85b0dcb7d294c3ddea69f392cba'}

## Model Comparison

The table below is sorted by validation AUC-ROC. Positive-class metrics refer to `diabetes_positive=1`; test metrics are final held-out results.

In [3]:
results = pd.read_csv(metrics_path)
display_columns = [
    "model",
    "sampling",
    "validation_auc_roc",
    "selection_threshold",
    "test_auc_roc",
    "test_recall_positive",
    "test_f1_positive",
    "test_false_negative",
    "test_true_positive",
]
display_table = results[display_columns].copy()
for column in display_table.select_dtypes(include="number").columns:
    display_table[column] = display_table[column].round(4)
display_table

,model,sampling,validation_auc_roc,selection_threshold,test_auc_roc,test_recall_positive,test_f1_positive,test_false_negative,test_true_positive
0,MLP 3 hidden layers (No SMOTE),No SMOTE,0.8471,0.3079,0.8113,0.8519,0.6866,8,46
1,MLP 1 hidden layer (SMOTE),SMOTE,0.8440,0.4546,0.8339,0.8519,0.7244,8,46
2,MLP 1 hidden layer (No SMOTE),No SMOTE,0.8378,0.3218,0.8263,0.7593,0.6777,13,41
3,MLP 7 hidden layers (No SMOTE),No SMOTE,0.8298,0.3517,0.7857,0.6852,0.6325,17,37
4,Logistic Regression (SMOTE),SMOTE,0.8281,0.5153,0.8230,0.6667,0.6316,18,36
5,MLP 7 hidden layers (SMOTE),SMOTE,0.8248,0.5883,0.8074,0.6111,0.6000,21,33
6,MLP 3 hidden layers (SMOTE),SMOTE,0.8245,0.5081,0.7917,0.6481,0.6034,19,35
7,Logistic Regression (No SMOTE),No SMOTE,0.8231,0.3281,0.8267,0.7778,0.7000,12,42
8,MLP 12 hidden layers (SMOTE),SMOTE,0.8142,0.2347,0.7615,0.8704,0.6438,7,47
9,MLP 12 hidden layers (No SMOTE),No SMOTE,0.8142,0.3754,0.7800,0.6481,0.6034,19,35


## Validation-Selected Model

The single-split selected model is chosen before looking at final test metrics: highest validation AUC-ROC wins for this train/validation/test split, and its validation-selected threshold is applied to test predictions.

In [4]:
summary["selected_by_validation_auc"]

{'model': 'MLP 3 hidden layers (No SMOTE)',
 'model_family': 'mlp',
 'sampling': 'No SMOTE',
 'hidden_layers': 3.0,
 'epochs_run': 22.0,
 'selection_threshold': 0.3079342842102051,
 'threshold_strategy': 'max_f1',
 'validation_accuracy': 0.7621621621621621,
 'validation_auc_roc': 0.8471074380165289,
 'validation_precision_positive': 0.6086956521739131,
 'validation_recall_positive': 0.875,
 'validation_f1_positive': 0.717948717948718,
 'validation_specificity_negative': 0.7024793388429752,
 'validation_true_negative': 85,
 'validation_false_positive': 36,
 'validation_false_negative': 8,
 'validation_true_positive': 56,
 'test_accuracy': 0.7272727272727273,
 'test_auc_roc': 0.8112962962962963,
 'test_precision_positive': 0.575,
 'test_recall_positive': 0.8518518518518519,
 'test_f1_positive': 0.6865671641791045,
 'test_specificity_negative': 0.66,
 'test_true_negative': 66,
 'test_false_positive': 34,
 'test_false_negative': 8,
 'test_true_positive': 46,
 'accuracy': 0.7272727272727273

## Test Audit

The audit entry records which model has the highest held-out test AUC-ROC. It is useful context, but it is not the model-selection rule.

In [5]:
summary["best_by_test_auc_for_audit"]

{'model': 'MLP 1 hidden layer (SMOTE)',
 'model_family': 'mlp',
 'sampling': 'SMOTE',
 'hidden_layers': 1.0,
 'epochs_run': 52.0,
 'selection_threshold': 0.4546493887901306,
 'threshold_strategy': 'max_f1',
 'validation_accuracy': 0.7567567567567568,
 'validation_auc_roc': 0.84400826446281,
 'validation_precision_positive': 0.6091954022988506,
 'validation_recall_positive': 0.828125,
 'validation_f1_positive': 0.7019867549668874,
 'validation_specificity_negative': 0.71900826446281,
 'validation_true_negative': 87,
 'validation_false_positive': 34,
 'validation_false_negative': 11,
 'validation_true_positive': 53,
 'test_accuracy': 0.7727272727272727,
 'test_auc_roc': 0.8338888888888888,
 'test_precision_positive': 0.6301369863013698,
 'test_recall_positive': 0.8518518518518519,
 'test_f1_positive': 0.7244094488188977,
 'test_specificity_negative': 0.73,
 'test_true_negative': 73,
 'test_false_positive': 27,
 'test_false_negative': 8,
 'test_true_positive': 46,
 'accuracy': 0.772727272

## Run the Experiment

From the repository root, run the short check or the full experiment:

```bash
python scripts/run_experiment.py --quick --no-plots --output-dir results/smoke
python scripts/run_experiment.py
```

Use `--threshold 0.5` only when a fixed threshold comparison is required.

## Caveat

This is a small academic experiment, not a clinical model. The Pima dataset is demographically narrow, so the results are best read as a model-comparison and evaluation-workflow exercise.